### MIDDLEWARE

MiddleWare provides a way to more tightly contril what happens inside the agent. Middleware is useful for the following:
- Tracking agent behavior with logging, analytics, and debugging.
- Transforming prompts, tool selection, and output formatting.
- Adding retires, fallbacks, and early termination logic.
- Applying rate limits, guardlrails, and PII detection.

In [7]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

### SUMMARIZATION MIDDLEWARE

Automatically sumarize conversation history when approaching token limits, preserving recent messages while compressing older context. Summarization is useful for the following:

- Long-running conversation that exceed context windows.
- Multi-turn dialogues with extensive history.
- Applicatiions where preserving full conversation context matters.

In [8]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

# Messages Based Summarization

agent = create_agent(
    model="groq:qwen/qwen3-32b",
    checkpointer = InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:qwen/qwen3-32b",
            trigger=("messages", 10),

            keep=("messages",4)
        )
    ]
)

In [9]:
# Run with thread ID
config = {"configurable":{"thread_id":"test-1"}}

In [10]:
# Alternate Test data
questions = [
    "What is 2+2?",
    "What is 8*3?",
    "What is 2+31?",
    "What is 32+22?",
    "What is 22*2?",
    "What is 22+2@?",
    "What is 53+21?",
]

for q in questions:
    response = agent.invoke({"messages":[HumanMessage(content = q)]}, config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")

Messages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='0ec96b40-d5fd-4904-b8d0-cb2d2586347f'), AIMessage(content="<think>\nOkay, so I need to figure out what 2 plus 2 is. Let me start by recalling basic arithmetic. Addition is one of the fundamental operations in math, right? When you add two numbers together, you're combining their quantities. So 2 and 2 are both whole numbers, and adding them should give a straightforward result.\n\nLet me visualize this. If I have two apples and someone gives me two more apples, how many apples do I have in total? It should be four, right? That's a simple way to think about it. Another way is using a number line. Starting at 2 and moving two units to the right lands me on 4. That also makes sense.\n\nWait, maybe I should break it down digit by digit. The number 2 is a single digit, and adding another 2 would just sum the digits. So 2 + 2 equals 4. There's no carrying over here because both number

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `qwen/qwen3-32b` in organization `org_01ktbvvmgrfd7r4e6anftkbzzd` service tier `on_demand` on tokens per minute (TPM): Limit 6000, Used 4637, Requested 1498. Please try again in 1.349999999s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

### BASED ON TOKEN SIZE

In [11]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str)->str:
    """Search Hotels - Returns Long response to use more token"""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 Star, $350/Night, SPA, Pool, GYM
    2. City Inn - 4 Star, $180/Night, Business Center
    3. Budget Stay - 3 Star, $75/Night, Free WiFi"""

agent = create_agent(
    model="groq:qwen/qwen3-32b",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:qwen/qwen3-32b",
            trigger=("tokens",550),
            keep=("tokens",200)
        )
    ]
)

config = {"configurable": {"thread_id" : "test-1"}}

#Token Counter (Approximate)
def count_token(messages):
    total_chrs = sum(len(str(m.content)) for m in messages)
    return total_chrs // 4  # 4 chars = 1 Token

In [ ]:
# RUN TEST

cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages":[HumanMessage(content = f"Find Hotels in {city}")]},
        config = config
    )

    tokens = count_token(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])}messages")
    print(f"{(response['messages'])}")

Paris: ~125 tokens, 4messages
[HumanMessage(content='Find Hotels in Paris', additional_kwargs={}, response_metadata={}, id='77500c39-da92-4361-a16b-92de0965bd4a'), AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking to find hotels in Paris. Let me check the tools provided. There\'s a function called search_hotels that takes a city parameter. The city here is Paris. I need to make sure the function is called correctly. The parameters require a city string, so I\'ll use "Paris" as the value. I should return a tool_call with the function name and the arguments in JSON. Let me structure that properly. No other tools are available, so this should be the only function call. Double-checking the required fields, everything seems in order. Time to format the response.\n', 'tool_calls': [{'id': 'd7gncjqdy', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 148,

### BASED ON FRACTION

In [6]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotel(city: str)->str:
    """Search Hotels"""
    return f"""Hotels in {city}: Grand Hotel $350, City Inn $180, Budget Stay $75"""

#LOW FRACTION FOR TESTING

agent = create_agent(
    model="groq:qwen/qwen3-32b",
    tools=[search_hotel],
    checkpointer=InMemorySaver(),
        middleware=[
            SummarizationMiddleware(model="groq:qwen/qwen3-32b",
        trigger=("fraction", 0.005), # 0.5% = ~640 Tokens
        keep=("fraction", 0.002),  # 0.2% = ~256 Tokens
    )
    
    ]
)

config = {"configurable": {"thread_id": "test-1"}}

#Token Counter
def count_token(messages):
    return sum(len(str(m.content)) for m in messages) // 4

#TEST

cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages":[HumanMessage(content = f"Find Hotels in {city}")]},
        config = config
    )

    tokens = count_token(response["messages"])
    fraction = tokens / 131072 # groq:qwen3-32b
    print(f"{city}: ~{tokens} tokens ({fraction:.4%}), {len(response['messages'])} msgs")
    print(f"{(response['messages'])}")



Paris: ~90 tokens (0.0687%), 4 msgs
[HumanMessage(content='Find Hotels in Paris', additional_kwargs={}, response_metadata={}, id='a22a9146-a692-465f-80f3-37a30189bc65'), AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user is asking to find hotels in Paris. Let me check the available tools. There's a function called search_hotel that requires a city parameter. The user specified Paris, so I need to call that function with the city set to Paris. I'll make sure the arguments are correctly formatted as JSON within the tool_call tags.\n", 'tool_calls': [{'id': '6eqht38rq', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotel'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 94, 'prompt_tokens': 147, 'total_tokens': 241, 'completion_time': 0.133406596, 'completion_tokens_details': {'reasoning_tokens': 69}, 'prompt_time': 0.006171151, 'prompt_tokens_details': None, 'queue_time': 0.046932649, 'total_time': 0.139577747}, '

### HUMAN IN THE LOOP MIDDLEWARE

Pause agent execution for human approval, editing, or rejection of tool calls before they execute. Human-in-loop is useful for the following:

- High-stakes operations requiring human approval (e.g. Database Write, Financial Transactions).
- Compliance workflows where human oversight is mandotory.
- Long-running conversation where human feedback guides the agent.

In [7]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

def read_email_tool(email_id: str)-> str:
    """Mock function to read an email by its ID."""
    return f"Email Content for the ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock Function to send an email"""
    return f"Email sent to {recipient} with Subject '{subject}'"


agent = create_agent(
    model="groq:qwen/qwen3-32b",
    tools=[read_email_tool, send_email_tool],
    checkpointer = InMemorySaver(),
    middleware = [
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]
                },
                "read_email_tool":False,
                }
            )
        ]
)


In [8]:
config = {"configurable": {"thread_id": "test-approve"}}

# STEP 1: REQUEST
result = agent.invoke(
    {"messages": [HumanMessage(content="Send Email to john@test.com with subject 'Hello' and body 'How are you?' ")]},
    config=config
)

In [9]:
print(result.get("__interrupt__"))

[Interrupt(value={'action_requests': [{'name': 'send_email_tool', 'args': {'body': 'How are you?', 'recipient': 'john@test.com', 'subject': 'Hello'}, 'description': "Tool execution requires approval\n\nTool: send_email_tool\nArgs: {'body': 'How are you?', 'recipient': 'john@test.com', 'subject': 'Hello'}"}], 'review_configs': [{'action_name': 'send_email_tool', 'allowed_decisions': ['approve', 'edit', 'reject']}]}, id='c045e3887546ae78a8039ebbea8195d8')]


In [10]:
# STEP 2: Approve
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")

result = agent.invoke(
    Command(
        resume={
            "decisions": [
                {
                    "type": "approve",
                    }
            ]
        }
    ),
    config=config,

)

print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: ✅ Successfully sent email to **john@test.com** with subject **"Hello"**. Let me know if you need to send or read any other emails!


In [11]:
result

{'messages': [HumanMessage(content="Send Email to john@test.com with subject 'Hello' and body 'How are you?' ", additional_kwargs={}, response_metadata={}, id='2f5ddee9-5bd0-4d17-a4e9-da9753fd27ae'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, let's see. The user wants to send an email to john@test.com with the subject 'Hello' and body 'How are you?'. I need to check the available tools to see which one can handle this.\n\nLooking at the tools provided, there's the send_email_tool. Its parameters are recipient, subject, and body, all required. The user provided all three: recipient is john@test.com, subject is 'Hello', and body is 'How are you?'. \n\nSo I should call the send_email_tool with these arguments. The other tool, read_email_tool, isn't needed here since the user isn't asking to read an email. Everything seems in order. Just need to format the tool_call correctly with the parameters.\n", 'tool_calls': [{'id': 'h7wtvvc0q', 'function': {'arguments': '

### Reject


In [12]:
config = {"configurable": {"thread_id": "test-reject"}}

#STEP 1: REQUEST

result = agent.invoke(
    {"messages": [HumanMessage(content="Send email tto john@gmail.com with subject 'Hello' and Boody 'How are you?' ")]},
    config = config
)

In [13]:
#STEP 2: REJCT

if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")

result = agent.invoke(
    Command(
        resume={
            "decisions": [
                {
                    "type": "reject",
                    }
            ]
        }
    ),
    config=config,

)

print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: The email request was not sent. The system requires you to explicitly request the action again if you'd like to proceed. Would you like me to send the email to john@gmail.com with the subject "Hello" and body "How are you?"?


In [14]:
result

{'messages': [HumanMessage(content="Send email tto john@gmail.com with subject 'Hello' and Boody 'How are you?' ", additional_kwargs={}, response_metadata={}, id='10872b27-08cd-4549-a27b-14a18a9b2513'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, let's see. The user wants to send an email to john@gmail.com with the subject 'Hello' and body 'How are you?'. I need to check which tool to use here. The available tools are read_email_tool and send_email_tool. Since the user is asking to send an email, the send_email_tool is the right choice.\n\nLooking at the parameters for send_email_tool, it requires recipient, subject, and body. The user provided all three: recipient is john@gmail.com, subject is 'Hello', and body is 'How are you?'. I should make sure all required parameters are included. Yep, all required fields are there. So I'll construct the tool call with these arguments. No need to use the read_email_tool here because the request is about sending, not rea

### EDITING

In [19]:
config = {"configurable": {"thread_id": "test-edit"}}

#STEP 1: REQUEST

result = agent.invoke(
    {"messages": [HumanMessage(content="Send email tto john@gmail.com with subject 'Hello' and Boody 'How are you?' ")]},
    config = config
)

In [20]:
#STEP 2: EDIT

if "__interrupt__" in result:
    print("⏸️ Paused! Editing...")

result = agent.invoke(
    Command(
        resume={
            "decisions": [
                {
                    "type": "edit",
                    "edited_action": {
                        "name": "send_email_tool",
                        "args": {
                            "recipient": "john@gmail.com",
                            "subject": "Hello",
                            "body": "How are you?"
                        },
                    },
                }
            ]
        }
    ),
    config=config,
)

print(f"✅ Result: {result['messages'][-1].content}")


⏸️ Paused! Editing...
✅ Result: The email with the subject "Hello" and body "How are you?" has been successfully sent to john@gmail.com. Let me know if you need further assistance!
